<a href="https://colab.research.google.com/github/tomerhan/Practices/blob/main/exr1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import os
os.chdir("/content/drive/MyDrive")

In [29]:
with open('students.txt', 'r') as file:
    contents = file.read()
    print(contents)

שם פרטי: תומר
שם משפחה: חנניה
דוא"ל: Tomer.Hananya@e.braude.ac.il
רשימה של דברים שלמדתי בסמסטר הזה: 
כריית נתונים ומערכות לומדות 
אוטומטים וחישוביות 
מבוא למחשוב ענן 
מבוא לבינה מלאכותית
טכנולוגיות אינטרנט מתקדמות (WEB) 
קישור מעניין
https://en.wikipedia.org/wiki/Skyhook_(structure)
תוכנית אהובה: 8678



In [21]:
def parse_student_data(content):
    student_data = {}
    lines = [line.strip() for line in content.split('\n') if line.strip()]

    field_map = {
        'שם פרטי': 'first_name',
        'שם משפחה': 'last_name',
        'דוא"ל': 'email',
        'רשימה של דברים שלמדתי בסמסטר הזה': 'courses',
        'קישור מעניין': 'interesting_link',
        'תוכנית אהובה': 'favorite_program' # New field
    }

    current_field = None
    courses_list = []

    i = 0
    while i < len(lines):
        line = lines[i]
        found_field_label = False

        for hebrew_label, english_key in field_map.items():
            if line.startswith(hebrew_label + ':'):
                student_data[english_key] = line.split(':', 1)[1].strip()
                current_field = english_key
                found_field_label = True
                break
            elif line == hebrew_label:
                if hebrew_label == 'קישור מעניין':
                    # The link is expected on the next line
                    if i + 1 < len(lines):
                        student_data[english_key] = lines[i+1].strip()
                        i += 1 # Consume the next line as it's the link
                    else:
                        student_data[english_key] = None # No link found
                    current_field = english_key
                    found_field_label = True
                    break
                elif hebrew_label == 'רשימה של דברים שלמדתי בסמסטר הזה':
                    current_field = english_key
                    found_field_label = True
                    # Courses start from the next line, no value on this header line
                    break

        if not found_field_label:
            if current_field == 'courses':
                courses_list.append(line)
        i += 1

    student_data['courses'] = courses_list
    return student_data

student_data = parse_student_data(contents)
print(student_data)

{'first_name': 'תומר', 'last_name': 'חנניה', 'email': 'Tomer.Hananya@e.braude.ac.il', 'courses': ['כריית נתונים ומערכות לומדות', 'אוטומטים וחישוביות', 'מבוא למחשוב ענן', 'מבוא לבינה מלאכותית', 'טכנולוגיות אינטרנט מתקדמות (WEB)'], 'interesting_link': 'https://en.wikipedia.org/wiki/Skyhook_(structure)'}


In [15]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create a list of student names for the dropdown
# Since student_data currently holds only one student, we'll create a list with that student's full name
student_names = [f"{student_data['first_name']} {student_data['last_name']}"]

print("ipywidgets modules imported and student names list created.")

ipywidgets modules imported and student names list created.


In [23]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create a list of student names for the dropdown
# Since student_data currently holds only one student, we'll create a list with that student's full name
student_names = [f"{student_data['first_name']} {student_data['last_name']}"]

dropdown = widgets.Dropdown(
    options=student_names,
    value=student_names[0] if student_names else None,
    description='Select Student:',
    disabled=False,
)

output_display = widgets.Output()

# Create widgets for displaying student details
first_name_widget = widgets.Text(description='First Name:', disabled=True)
last_name_widget = widgets.Text(description='Last Name:', disabled=True)
email_widget = widgets.Text(description='Email:', disabled=True)
courses_textarea = widgets.Textarea(description='Courses:', rows=5, disabled=True)
interesting_link_widget = widgets.Text(description='Interesting Link:', disabled=True)

# Create widgets for 'Favorite Program' input and button
favorite_program_input = widgets.Text(description='Favorite Program:')
update_button = widgets.Button(description='עדכן תוכנית אהובה')

def update_student_info(change):
    with output_display:
        output_display.clear_output()
        if change['new']:
            selected_student_name = change['new']
            # Assuming student_data corresponds to the selected student
            # For now, we only have one student in student_data
            # In a real scenario, you'd look up the data based on the selected name

            first_name_widget.value = student_data['first_name']
            last_name_widget.value = student_data['last_name']
            email_widget.value = student_data['email']
            courses_textarea.value = "\n".join(student_data['courses'])
            interesting_link_widget.value = student_data['interesting_link']
            favorite_program_input.value = student_data.get('favorite_program', '')


# Observe the dropdown for changes
dropdown.observe(update_student_info, names='value')

# Display the form
form = widgets.VBox([dropdown, output_display])
display(form)

# Call the update function once initially to populate the display
# This simulates a change event for the initial value
update_student_info({'new': dropdown.value})

In [24]:
def on_update_button_click(b):
    with output_display:
        output_display.clear_output()
        new_program_value = favorite_program_input.value

        # Read current content
        with open('students.txt', 'r', encoding='utf-8') as file:
            current_file_content = file.read()

        # Construct new favorite program line
        new_program_line = f"תוכנית אהובה: {new_program_value}"

        # Remove any existing 'favorite_program' line before appending the new one
        # This handles cases where the field already exists or is updated multiple times
        lines = current_file_content.split('\n')
        filtered_lines = [line for line in lines if not line.strip().startswith('תוכנית אהובה:')]
        modified_content = '\n'.join(filtered_lines).strip() + '\n' + new_program_line + '\n'

        # Write modified content back to the file
        with open('students.txt', 'w', encoding='utf-8') as file:
            file.write(modified_content)

        # Re-read the updated students.txt file
        global contents
        with open('students.txt', 'r', encoding='utf-8') as file:
            contents = file.read()

        # Re-parse the updated contents
        global student_data
        student_data = parse_student_data(contents)

        # Refresh the displayed student information
        update_student_info({'new': dropdown.value})

In [25]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create a list of student names for the dropdown
# Since student_data currently holds only one student, we'll create a list with that student's full name
student_names = [f"{student_data['first_name']} {student_data['last_name']}"]

dropdown = widgets.Dropdown(
    options=student_names,
    value=student_names[0] if student_names else None,
    description='Select Student:',
    disabled=False,
)

output_display = widgets.Output()

# Create widgets for displaying student details
first_name_widget = widgets.Text(description='First Name:', disabled=True)
last_name_widget = widgets.Text(description='Last Name:', disabled=True)
email_widget = widgets.Text(description='Email:', disabled=True)
courses_textarea = widgets.Textarea(description='Courses:', rows=5, disabled=True)
interesting_link_widget = widgets.Text(description='Interesting Link:', disabled=True)

# Create widgets for 'Favorite Program' input and button
favorite_program_input = widgets.Text(description='Favorite Program:')
update_button = widgets.Button(description='עדכן תוכנית אהובה')

def update_student_info(change):
    with output_display:
        output_display.clear_output()
        if change['new']:
            selected_student_name = change['new']
            # Assuming student_data corresponds to the selected student
            # For now, we only have one student in student_data
            # In a real scenario, you'd look up the data based on the selected name

            first_name_widget.value = student_data['first_name']
            last_name_widget.value = student_data['last_name']
            email_widget.value = student_data['email']
            courses_textarea.value = "\n".join(student_data['courses'])
            interesting_link_widget.value = student_data['interesting_link']
            favorite_program_input.value = student_data.get('favorite_program', '')


# Observe the dropdown for changes
dropdown.observe(update_student_info, names='value')

# Attach the update function to the button click event
update_button.on_click(on_update_button_click)

# Display the form
form = widgets.VBox([
    dropdown,
    first_name_widget,
    last_name_widget,
    email_widget,
    courses_textarea,
    interesting_link_widget,
    favorite_program_input,
    update_button,
    output_display
])
display(form)

# Call the update function once initially to populate the display
# This simulates a change event for the initial value
update_student_info({'new': dropdown.value})